<a href="https://colab.research.google.com/github/valsson-group/UNT-ChemicalApplicationsOfMachineLearning-Spring2026/blob/main/Lecture-24-April-23-2026/Lecture-24_AtomAndBondFeatureAnalysis_MeltingPoint.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Lecture 24 - Atom and Bond Feature Analysis - Melting Point Dataset



In [1]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.0/37.0 MB 46.0 MB/s eta 0:00:00


Import all basic pacakges

In [2]:
# basic
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# RDkit
from rdkit import Chem
from rdkit.Chem import rdMolDescriptors

# For progress bar
from tqdm.auto import tqdm

tqdm.pandas()

Download dataset

In [3]:
# Bash script to download all the dataset. Don't worry if you don't understand it
%%bash

url="https://raw.githubusercontent.com/valsson-group/UNT-ChemicalApplicationsOfMachineLearning-Spring2026/refs/heads/main/Assignment-2/"
dataset_filename="BradleyDoublePlusGoodMeltingPointDataset.csv"

rm -f ${dataset_filename}

wget ${url}/${dataset_filename} &> /dev/null

ls

BradleyDoublePlusGoodMeltingPointDataset.csv
sample_data


## Read dataset

In [4]:
data_mp = pd.read_csv("BradleyDoublePlusGoodMeltingPointDataset.csv")
data_mp = data_mp.drop(columns=['csid','link','source','count','min','max','range'])

In [14]:
explict_hydrogens = True
molecules = []

for smiles in tqdm(data_mp['smiles']):
  mol = Chem.MolFromSmiles(smiles)
  if mol:
    if explict_hydrogens:
      mol = Chem.AddHs(mol)
    molecules.append(mol)


  0%|          | 0/3041 [00:00<?, ?it/s]

[02:56:50] Can't kekulize mol.  Unkekulized atoms: 0 1 2 3 4
[02:56:50] Can't kekulize mol.  Unkekulized atoms: 2 3 4 5 6
[02:56:50] Can't kekulize mol.  Unkekulized atoms: 24 25 26 27 28 31 32 33 34
[02:56:50] Can't kekulize mol.  Unkekulized atoms: 0 1 2 3 4
[02:56:50] Can't kekulize mol.  Unkekulized atoms: 0 1 2 3 4 5 6 7 8
[02:56:50] Can't kekulize mol.  Unkekulized atoms: 1 2 3 4 5 6 7 8 9
[02:56:50] Can't kekulize mol.  Unkekulized atoms: 16 17 18 19 20 21 22 23 24
[02:56:50] Can't kekulize mol.  Unkekulized atoms: 3 4 5 6 7 8 9 10 11
[02:56:50] Can't kekulize mol.  Unkekulized atoms: 3 4 5 6 8
[02:56:50] Can't kekulize mol.  Unkekulized atoms: 0 1 2 3 4 5 6 7 8
[02:56:50] Can't kekulize mol.  Unkekulized atoms: 1 2 3 4 5 6 7 8 9
[02:56:50] Can't kekulize mol.  Unkekulized atoms: 0 1 2 3 4 5 6 7 8
[02:56:50] Can't kekulize mol.  Unkekulized atoms: 0 1 2 3 12 13 14 15 16
[02:56:50] Can't kekulize mol.  Unkekulized atoms: 0 1 2 3 4 5 6 7 8
[02:56:50] Can't kekulize mol.  Unkekuliz

## Look at atom and bond properties

In [15]:
atomic_properties = {'atomic_num': [],
                     'degree' : [],
                     'formal_charge': [],
                     'chiral_tag': [],
                     'chiral_tag_int': [],
                     'num_Hs': [],
                     'hybridization': [],
                     'hybridization_int': [],
                     'is_aromatic': [],
                     'is_aromatic_int': [],
                     'mass': []
                     }

atomic_functions = {'atomic_num': lambda x: x.GetAtomicNum(),
                    'degree': lambda x: x.GetDegree(),
                    'formal_charge': lambda x: x.GetFormalCharge(),
                    'chiral_tag': lambda x: x.GetChiralTag(),
                    'chiral_tag_int': lambda x: int(x.GetChiralTag()),
                    'num_Hs': lambda x: x.GetTotalNumHs(),
                    'hybridization': lambda x: x.GetHybridization(),
                    'hybridization_int': lambda x: int(x.GetHybridization()),
                    'is_aromatic': lambda x: x.GetIsAromatic(),
                    'is_aromatic_int': lambda x: int(x.GetIsAromatic()),
                    'mass': lambda x: x.GetMass()
                    }

bond_properties = {'bond_type': [],
                   'bond_type_int': [],
                   'stereo': [],
                   'stereo_int': [],
                   'is_conjugated': [],
                   'is_conjugated_int': [],
                   'is_in_ring': [],
                   'is_in_ring_int': []
                  }

bond_functions = {'bond_type': lambda x: x.GetBondType(),
                  'bond_type_int': lambda x: int(x.GetBondType()),
                  'stereo': lambda x: x.GetStereo(),
                  'stereo_int': lambda x: int(x.GetStereo()),
                  'is_conjugated': lambda x: x.GetIsConjugated(),
                  'is_conjugated_int': lambda x: int(x.GetIsConjugated()),
                  'is_in_ring': lambda x: x.IsInRing(),
                  'is_in_ring_int': lambda x: int(x.IsInRing())
                 }

for mol in tqdm(molecules):
    if mol:
      for atom in mol.GetAtoms():
        for key in atomic_properties.keys():
          atomic_properties[key].append(atomic_functions[key](atom))
      for bond in mol.GetBonds():
        for key in bond_properties.keys():
          bond_properties[key].append(bond_functions[key](bond))


  0%|          | 0/3025 [00:00<?, ?it/s]

In [16]:
from collections import Counter

print("Atom Properties:")
for key in atomic_properties.keys():
  print(f"- {key}")
  counter = Counter(atomic_properties[key])
  for k, v in counter.items():
    print(f"  - {k}: {v}")
  print("#")
print("#----------")
print(" ")

print("Bond Properties:")
for key in bond_properties.keys():
  print(f"- {key}")
  counter = Counter(bond_properties[key])
  for k, v in counter.items():
    print(f"  - {k}: {v}")
  print("#")
print("#----------")


Atom Properties:
- atomic_num
  - 6: 27557
  - 1: 33607
  - 8: 4463
  - 7: 1946
  - 9: 461
  - 16: 296
  - 35: 304
  - 17: 817
  - 53: 71
  - 5: 5
  - 15: 33
  - 14: 35
#
- degree
  - 4: 9710
  - 1: 37779
  - 2: 2945
  - 3: 19161
#
- formal_charge
  - 0: 69027
  - -1: 284
  - 1: 284
#
- chiral_tag
  - CHI_UNSPECIFIED: 69339
  - CHI_TETRAHEDRAL_CCW: 131
  - CHI_TETRAHEDRAL_CW: 125
#
- chiral_tag_int
  - 0: 69339
  - 2: 131
  - 1: 125
#
- num_Hs
  - 0: 69595
#
- hybridization
  - SP3: 12266
  - UNSPECIFIED: 33607
  - SP2: 23348
  - SP: 374
#
- hybridization_int
  - 4: 12266
  - 0: 33607
  - 3: 23348
  - 2: 374
#
- is_aromatic
  - False: 53435
  - True: 16160
#
- is_aromatic_int
  - 0: 53435
  - 1: 16160
#
- mass
  - 12.011: 27557
  - 1.008: 33607
  - 15.999: 4463
  - 14.007: 1946
  - 18.998: 461
  - 32.067: 296
  - 79.904: 304
  - 35.453: 817
  - 126.904: 71
  - 10.812: 5
  - 30.974: 33
  - 28.086: 35
#
#----------
 
Bond Properties:
- bond_type
  - SINGLE: 50852
  - TRIPLE: 174
  - DOUB